# Lecture 2: Data Preprocessing Practice
## Boston apartment listings: from a messy export to a table you can trust
**Salem Othman · Student notebook**

In Lecture 1 you learned the preprocessing tools. Today you use them on new data and **defend every decision**.

**The question:** what does an apartment near WIT cost?

You have two tables:
- `listings_raw`: 15 rows exported from a listing website
- `neighborhoods`: walking time from each neighborhood to WIT

You will build a clean table for a short report and a number-only table for a rent model later. You will **not** train a model today.

### Today's six steps
| Part | Step | What you do | Exercise |
|---|---|---|---|
| 1 | Inspect | Find problems before changing anything | E1 |
| 2 | Integrate | Join the two tables and check who didn't match | E2 |
| 3 | Clean | Turn text into numbers, blank impossible values, fix duplicates | E3 |
| 4 | Missing values | Fill apartment size with a sensible value and keep a flag | E4 |
| 5 | Reduce | Drop columns that add nothing | E5 |
| 6 | Transform | Encode neighborhoods, split, and scale | E6 |

**For every exercise:** predict → write the code → check → explain.

**How to work**
- Run cells from top to bottom with **Shift + Enter**.
- In an exercise cell, **write the code yourself** under each numbered step. The steps tell you what to build and give a hint. Keep the variable names exactly as written; the checks look for them.
- Run the **Check** cell under each exercise. `PASS` means the step is done. `FIX` tells you what to look at.
- Type your written answers in the *Your answer* cells (double-click to edit).

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
RANDOM_STATE = 42

def check(label, passed):
    """Print PASS or FIX for one requirement."""
    print(("PASS  " if passed else "FIX   ") + label)

def need(value, step):
    """Stop with a clear message if an earlier step is not finished."""
    if value is None:
        raise RuntimeError(f"Finish {step} first, then run this cell again.")

print("Setup complete.")

Setup complete.


# Part 1 · Inspect
Look before you change anything. Every problem you report needs **evidence**: a column, a listing ID, and the value.

| Column | Meaning |
|---|---|
| `listing_id` | ID from the website. A re-posted apartment keeps its ID. |
| `posted` | Date the listing was posted |
| `neighborhood` | Neighborhood name, as typed by the landlord |
| `bedrooms` | Studio, 1, 2, or 3 |
| `rent_text` | Monthly rent, as shown on the website |
| `sqft` | Apartment size in square feet |
| `city` | City |
| `size_m2` | Apartment size in square meters |
| `walk_minutes` | *(in `neighborhoods`)* Walking time to WIT |

In [2]:
listings_raw = pd.DataFrame({
    "listing_id":   ["L01", "L02", "L03", "L04", "L05", "L06", "L07", "L08",
                     "L09", "L10", "L11", "L12", "L07", "L11", "L13"],
    "posted":       ["2026-08-03", "2026-08-05", "2026-08-06", "2026-08-06", "2026-08-07",
                     "2026-08-08", "2026-08-01", "2026-08-09", "2026-08-10", "2026-08-11",
                     "2026-08-12", "2026-08-12", "2026-08-20", "2026-08-12", "2026-08-14"],
    "neighborhood": ["Mission Hill", "mission hill", "Fenway", "FENWAY ", "Back Bay",
                     "Allston", "JP", "Brighton", "Jamaica Plain", "Allston ",
                     "Back Bay", "Mission Hill", "Jamaica Plain", "Back Bay", "Fenway"],
    "bedrooms":     ["2", "Studio", "1", "2", "3", "2", "3", "1",
                     "studio", "1", "1", "3", "3", "1", "Studio"],
    "rent_text":    ["$2,900", "$1,850", "$2,450", "3,200", "$9,500", "$2,600", "$3,400", "$2,100",
                     "$0", "$2,050", "$3,300", "$3,600", "$3,150", "$3,300", "$2,000"],
    "sqft":         [750, 400, 520, np.nan, 1800, 800, 1100, 600,
                     350, 0, 550, np.nan, 1100, 550, np.nan],
    "city":         ["Boston"] * 15,
    "size_m2":      [69.7, 37.2, 48.3, np.nan, 167.2, 74.3, 102.2, 55.7,
                     32.5, 0.0, 51.1, np.nan, 102.2, 51.1, np.nan],
})

neighborhoods = pd.DataFrame({
    "neighborhood": ["Mission Hill", "Fenway", "Back Bay", "Allston", "Jamaica Plain", "Roxbury"],
    "walk_minutes": [10, 12, 25, 45, 30, 20],
})

display(listings_raw)
display(neighborhoods)

,listing_id,posted,neighborhood,bedrooms,rent_text,sqft,city,size_m2
0,L01,2026-08-03,Mission Hill,2,"$2,900",750.0,Boston,69.7
1,L02,2026-08-05,mission hill,Studio,"$1,850",400.0,Boston,37.2
2,L03,2026-08-06,Fenway,1,"$2,450",520.0,Boston,48.3
3,L04,2026-08-06,FENWAY,2,"3,200",NaN,Boston,NaN
4,L05,2026-08-07,Back Bay,3,"$9,500",1800.0,Boston,167.2
5,L06,2026-08-08,Allston,2,"$2,600",800.0,Boston,74.3
6,L07,2026-08-01,JP,3,"$3,400",1100.0,Boston,102.2
7,L08,2026-08-09,Brighton,1,"$2,100",600.0,Boston,55.7
8,L09,2026-08-10,Jamaica Plain,studio,$0,350.0,Boston,32.5
9,L10,2026-08-11,Allston,1,"$2,050",0.0,Boston,0.0


,neighborhood,walk_minutes
0,Mission Hill,10
1,Fenway,12
2,Back Bay,25
3,Allston,45
4,Jamaica Plain,30
5,Roxbury,20


In [3]:
print("Rows, columns:", listings_raw.shape)
display(listings_raw.dtypes)

Rows, columns: (15, 8)


,0
listing_id,object
posted,object
neighborhood,object
bedrooms,object
rent_text,object
sqft,float64
city,object
size_m2,float64


`rent_text` and `bedrooms` are stored as **text** (pandas shows `object` or `str`, depending on your version). You cannot average text, so they must become numbers before any math.

### E1 · Find the evidence
**Predict:** before running any code, look at the table above. Which column worries you most?

**Do:**
1. `missing_counts`: the number of blanks in each column
2. `label_counts`: how often each `neighborhood` label appears
3. `repeated_ids`: every row whose `listing_id` appears more than once

In [4]:
missing_counts = listings_raw.isna().sum()
label_counts = listings_raw["neighborhood"].value_counts()
repeated_ids = listings_raw[listings_raw.duplicated(subset="listing_id", keep=False)]

# 4. Display all three.
display("Missing Counts:")
display(missing_counts)
display("Label Counts:")
display(label_counts)
display("Repeated IDs:")
display(repeated_ids)

'Missing Counts:'

,0
listing_id,0
posted,0
neighborhood,0
bedrooms,0
rent_text,0
sqft,3
city,0
size_m2,3


'Label Counts:'

,count
neighborhood,
Back Bay,3
Mission Hill,2
Jamaica Plain,2
Fenway,2
mission hill,1
FENWAY,1
Allston,1
JP,1
Brighton,1


'Repeated IDs:'

,listing_id,posted,neighborhood,bedrooms,rent_text,sqft,city,size_m2
6,L07,2026-08-01,JP,3,"$3,400",1100.0,Boston,102.2
10,L11,2026-08-12,Back Bay,1,"$3,300",550.0,Boston,51.1
12,L07,2026-08-20,Jamaica Plain,3,"$3,150",1100.0,Boston,102.2
13,L11,2026-08-12,Back Bay,1,"$3,300",550.0,Boston,51.1


**Check E1**

In [5]:
need(missing_counts, "E1"); need(label_counts, "E1"); need(repeated_ids, "E1")
check("missing_counts has one number per column", set(missing_counts.index) == set(listings_raw.columns))
check("label_counts covers every row", label_counts.sum() == len(listings_raw))
check("repeated_ids holds exactly the rows with a repeated ID",
      repeated_ids.index.equals(listings_raw.index[listings_raw["listing_id"].duplicated(keep=False)]))

PASS  missing_counts has one number per column
PASS  label_counts covers every row
PASS  repeated_ids holds exactly the rows with a repeated ID


**Write:** list four problems. For each, give the column, a listing ID, and the value.

*Your answer:*
1. Neighborhood, L07, JP and Jamaca Plain. Different variations of the same information
2. rent_text, 07, one has 3400 and one has 3150
3. posted, L07, Different dates, one is 08-01 and the other is 08-20
4. sqft ,L04, NaN instead of float64

# Part 2 · Integrate
We join `listings_raw` to `neighborhoods` on the `neighborhood` column to give each listing its walking time.

**Predict:** if we join on the labels exactly as typed, how many of the 15 rows will get **no** walking time?

*Your prediction:* 7

In [6]:
raw_join = listings_raw.merge(neighborhoods, on="neighborhood", how="left", indicator=True)
lost = raw_join[raw_join["_merge"] == "left_only"]
print("Rows with no walking time:", len(lost))
display(lost[["listing_id", "neighborhood"]])

Rows with no walking time: 5


,listing_id,neighborhood
1,L02,mission hill
3,L04,FENWAY
6,L07,JP
7,L08,Brighton
9,L10,Allston


The join ran with **no error**, yet it lost walking times. To Python, `"mission hill"` and `"Mission Hill"` are different text. **Clean the key column before you join.**

### E2 · Clean the key, then audit the join
`name_map` gives one standard name per neighborhood. `"jp"` is a common short form of Jamaica Plain.

1. `keyed`: a copy of `listings_raw` with `neighborhood` stripped of spaces, lowercased, and mapped
2. `unmapped`: rows whose label is not in `name_map` (should be empty)
3. `audit`: an **outer** join with `indicator=True`; show the `_merge` counts and the rows that did not match
4. `joined`: a **left** join. This is the table we keep.
5. `no_walk_ids`: listing IDs in `joined` that still have no walking time

`validate="many_to_one"` means many listings may share a neighborhood, but each neighborhood may appear only **once** in `neighborhoods`. If that is not true, pandas stops with an error.

In [7]:
name_map = {
    "mission hill": "Mission Hill", "fenway": "Fenway", "back bay": "Back Bay",
    "allston": "Allston", "brighton": "Brighton",
    "jamaica plain": "Jamaica Plain", "jp": "Jamaica Plain",
}

keyed = listings_raw.copy()
keyed["neighborhood"] = keyed["neighborhood"].str.strip().str.lower().map(name_map)
unmapped = keyed[keyed["neighborhood"].isna()]
print("Unmapped rows count:", len(unmapped))

audit = keyed.merge(neighborhoods, on="neighborhood", how="outer", indicator=True, validate="many_to_one")
print("Audit merge counts:")
print(audit["_merge"].value_counts())
display(audit[audit["_merge"] != "both"])

joined = keyed.merge(neighborhoods, on="neighborhood", how="left", indicator=True ,validate="many_to_one")

no_walk_ids = joined.loc[joined["walk_minutes"].isna(), "listing_id"].tolist()
print("No walk IDs:", no_walk_ids)

# Write your code below. Your code must create the variables set to None above.

# 1. keyed: a copy of listings_raw. Then replace its "neighborhood" column with a cleaned
#    version: remove spaces at both ends, make it lowercase, then map it with name_map.
#    Hint: .copy(), .str.strip(), .str.lower(), .map()

# 2. unmapped: the rows of keyed whose neighborhood is now blank. Print how many there are.
#    Hint: .isna()

# 3. audit: an OUTER join of keyed and neighborhoods on "neighborhood",
#    with indicator=True and validate="many_to_one".
#    Display the counts of the "_merge" column, then the rows where "_merge" is not "both".
#    Hint: keyed.merge(...), .value_counts()

# 4. joined: a LEFT join of keyed and neighborhoods (same key, same validate).

# 5. no_walk_ids: a Python list of the listing_id values in joined whose walk_minutes is blank.
#    Print it.
#    Hint: joined.loc[<condition>, "listing_id"].tolist()


Unmapped rows count: 0
Audit merge counts:
_merge
both          14
left_only      1
right_only     1
Name: count, dtype: int64


,listing_id,posted,neighborhood,bedrooms,rent_text,sqft,city,size_m2,walk_minutes,_merge
5,L08,2026-08-09,Brighton,1,"$2,100",600.0,Boston,55.7,NaN,left_only
15,NaN,NaN,Roxbury,NaN,NaN,NaN,NaN,NaN,20.0,right_only


No walk IDs: ['L08']


**Check E2**

In [8]:
need(keyed, "E2"); need(audit, "E2"); need(joined, "E2"); need(no_walk_ids, "E2")
check("every neighborhood label was mapped", keyed["neighborhood"].notna().all())
check("joined keeps every listing row: none lost, none added", len(joined) == len(listings_raw))
check("no_walk_ids matches the rows with a blank walking time",
      set(no_walk_ids) == set(joined.loc[joined["walk_minutes"].isna(), "listing_id"]))
check("cleaning the key fixed some of the lost rows", len(no_walk_ids) < len(lost))

PASS  every neighborhood label was mapped
PASS  joined keeps every listing row: none lost, none added
PASS  no_walk_ids matches the rows with a blank walking time
PASS  cleaning the key fixed some of the lost rows


**Write:** one listing still has no walking time, and one neighborhood has no listings. Is either one an error? What should the report say about that listing?

*Your answer:*
Only one of our rows is missing walking_minutes. It may be an error, but it still tells a valuable story in our data

### Worked example · A join can add rows
Suppose a second source adds another Fenway row to the lookup table, with 14 minutes instead of 12. Watch the row count.

In [9]:
need(keyed, "E2")
neighborhoods_bad = pd.concat(
    [neighborhoods, pd.DataFrame({"neighborhood": ["Fenway"], "walk_minutes": [14]})],
    ignore_index=True,
)
bigger = keyed.merge(neighborhoods_bad, on="neighborhood", how="left")
print("Rows before:", len(keyed), "| Rows after:", len(bigger))
display(bigger[bigger["neighborhood"] == "Fenway"][["listing_id", "neighborhood", "walk_minutes"]])

try:
    keyed.merge(neighborhoods_bad, on="neighborhood", how="left", validate="many_to_one")
except pd.errors.MergeError as error:
    print("validate stopped the join:", str(error).splitlines()[0])

Rows before: 15 | Rows after: 18


,listing_id,neighborhood,walk_minutes
2,L03,Fenway,12.0
3,L03,Fenway,14.0
4,L04,Fenway,12.0
5,L04,Fenway,14.0
16,L13,Fenway,12.0
17,L13,Fenway,14.0


validate stopped the join: Merge keys are not unique in right dataset; not a many-to-one merge


Every Fenway listing now appears twice, once with each walking time. `validate` turns this silent problem into an error you can see. Fix the lookup table; don't delete rows from the result.

# Part 3 · Clean
State the rule first, then change the data, then check the result.

| Problem | Rule for this lab |
|---|---|
| Rent stored as text (`"$2,450"`) | Remove `$` and `,`, then convert to a number |
| Bedrooms stored as text (`"Studio"`) | Studio = 0; the others keep their number |
| Rent of 0 or less | Invalid: blank it and flag it |
| Size of 0 sq ft or less | Invalid: blank it and flag it |

These rules fit this dataset. Another dataset may need different ones.

### E3 · Text to numbers, and blank the impossible values
Start from `joined`.
1. `cleaned["rent"]`: numbers made from `rent_text`
2. `cleaned["bedrooms"]`: numbers, with studio as 0
3. `rent_was_invalid` and `sqft_was_invalid`: `True` where the value is 0 or less
4. Replace those invalid values with `np.nan`

In [10]:
cleaned = None

need(joined, "E2")

# Write your code below. Your code must create the variable set to None above.

# 1. cleaned: a copy of joined.
cleaned = joined.copy()

# 2. cleaned["rent"]: start from rent_text, remove "$" and ",", remove spaces at both ends,
#    then convert to a number.
#    Hint: .str.replace("$", "", regex=False), .str.strip(), pd.to_numeric()
cleaned["rent"] = cleaned["rent_text"].str.replace("$", "", regex=False).str.replace(",", "", regex=False).str.strip()
cleaned["rent"] = pd.to_numeric(cleaned["rent"])

# 3. cleaned["bedrooms"]: remove spaces, make lowercase, change "studio" to "0",
#    then convert to whole numbers.
#    Hint: .replace({"studio": "0"}), .astype(int)
cleaned["bedrooms"] = cleaned["bedrooms"].str.lower().replace({"studio": "0"}).astype(int)

# 4. Two flag columns:
#    cleaned["rent_was_invalid"] is True where rent is 0 or less.
#    cleaned["sqft_was_invalid"] is True where sqft is 0 or less.
cleaned["rent_was_invalid"] = cleaned["rent"] <= 0
cleaned["sqft_was_invalid"] = cleaned["sqft"] <= 0


# 5. Set the invalid rent values and the invalid sqft values to np.nan.
#    Hint: cleaned.loc[<flag column>, "rent"] = np.nan
cleaned.loc[cleaned["rent_was_invalid"], "rent"] = np.nan
cleaned.loc[cleaned["sqft_was_invalid"], "sqft"] = np.nan

# 6. Display listing_id, rent_text, rent, bedrooms, sqft, and the two flags.
display(cleaned[["listing_id", "rent_text", "rent", "bedrooms", "sqft", "rent_was_invalid", "sqft_was_invalid"]])

,listing_id,rent_text,rent,bedrooms,sqft,rent_was_invalid,sqft_was_invalid
0,L01,"$2,900",2900.0,2,750.0,False,False
1,L02,"$1,850",1850.0,0,400.0,False,False
2,L03,"$2,450",2450.0,1,520.0,False,False
3,L04,"3,200",3200.0,2,NaN,False,False
4,L05,"$9,500",9500.0,3,1800.0,False,False
5,L06,"$2,600",2600.0,2,800.0,False,False
6,L07,"$3,400",3400.0,3,1100.0,False,False
7,L08,"$2,100",2100.0,1,600.0,False,False
8,L09,$0,NaN,0,350.0,True,False
9,L10,"$2,050",2050.0,1,NaN,False,True


**Check E3**

In [11]:
need(cleaned, "E3")
check("rent is a number column", pd.api.types.is_numeric_dtype(cleaned["rent"]))
check("bedrooms is a number column", pd.api.types.is_numeric_dtype(cleaned["bedrooms"]))
check("no rent or size of 0 or less remains",
      not (cleaned["rent"] <= 0).any() and not (cleaned["sqft"] <= 0).any())
check("every blank rent is explained by the invalid flag",
      cleaned["rent"].isna().sum() == cleaned["rent_was_invalid"].sum())
check("new blank sizes match the invalid flag",
      cleaned["sqft"].isna().sum() == joined["sqft"].isna().sum() + cleaned["sqft_was_invalid"].sum())

PASS  rent is a number column
PASS  bedrooms is a number column
PASS  no rent or size of 0 or less remains
PASS  every blank rent is explained by the invalid flag
PASS  new blank sizes match the invalid flag


**Write:** listing L05 rents for $9,500. Is that invalid under our rules? Should we keep it? Give one reason.

*Your answer:*
It is not invalid, it's rent is greater than 0 and sqft is greater than 0

### Worked example · Two kinds of duplicates
**Predict:** listing L07 appears twice: posted Aug 1 at \$3,400 and Aug 20 at \$3,150. Which row describes the apartment today?

*Your prediction:*
The one with Aug 20th as the date.

In [12]:
deduped = None
need(cleaned, "E3")
cleaned["posted"] = pd.to_datetime(cleaned["posted"])
display(cleaned[cleaned["listing_id"].duplicated(keep=False)]
        .sort_values("listing_id")[["listing_id", "posted", "neighborhood", "rent", "sqft"]])

# Kind 1: an exact copy (every column identical). Safe to remove.
no_copies = cleaned.drop_duplicates()
print("Exact copies removed:", len(cleaned) - len(no_copies))

# Kind 2: same ID, different values. Rule: keep the most recent post (the current price).
deduped = (no_copies.sort_values("posted")
                    .drop_duplicates(subset="listing_id", keep="last")
                    .sort_values("listing_id")
                    .reset_index(drop=True))
print("Listings left:", len(deduped))
display(deduped[deduped["listing_id"] == "L07"][["listing_id", "posted", "rent"]])

,listing_id,posted,neighborhood,rent,sqft
6,L07,2026-08-01,Jamaica Plain,3400.0,1100.0
12,L07,2026-08-20,Jamaica Plain,3150.0,1100.0
10,L11,2026-08-12,Back Bay,3300.0,550.0
13,L11,2026-08-12,Back Bay,3300.0,550.0


Exact copies removed: 1
Listings left: 13


,listing_id,posted,rent
6,L07,2026-08-20,3150.0


**Compare with Lecture 1.** There we used `keep="first"`. That table had no post date, so any single rule was a guess. Here, `keep="first"` would keep the **old** \$3,400 price. The date gives us a reason to choose.

### AI check 1 · The duplicate listing
Ask your AI tool:
> My apartment listings table has two rows with listing ID L07. One was posted Aug 1 at \$3,400. The other was posted Aug 20 at \$3,150. Everything else is the same. Which row should I keep, and why?

Paste its main recommendation (one or two sentences), then judge it with the class.

| Question | Your note |
|---|---|
| Did it use the post dates? | Yes |
| Did it give a rule you could apply to every listing? | Yes |
| Did it say what could go wrong? | Yes, a price drop |
| Did it invent anything, such as averaging the two prices? | No, it did not |

*AI answer:*
You should keep the row posted on August 20 at $3,150. In data preprocessing, when you have duplicate records with the same ID, the standard practice is to retain the most recent entry, as it likely reflects the most accurate and up-to-date information—in this case, a price drop.

Why Keep the Latest Record?
When resolving duplicate entries in a dataset, keeping the most recent record is generally the best approach for several reasons:
Data Accuracy The later post (August 20) likely represents a deliberate update to the listing, such as a price reduction to attract more interest.
Relevance If you are analyzing current market trends, the most recent price ($3,150) is the active price that potential renters will see.
Consistency Establishing a rule to always keep the latest record helps maintain a clean and standardized dataset moving forward.

# Part 4 · Missing values
**Rent:** listing L09 now has no rent. **Do not fill it.** Rent is what our report and future model are about, so a filled rent would be a made-up price. L09 stays in the table with rent unknown.

**Size:** four listings have no size. Below are three choices, using our own numbers.

**Predict:** listing L13 is a studio with no size. Which choice would give it a size bigger than any studio in our data?

*Your prediction:*
overall median

In [13]:
need(deduped, "the Part 3 duplicates example")
overall_median = deduped["sqft"].median()
bedroom_median = deduped.groupby("bedrooms")["sqft"].transform("median")

options = deduped.loc[deduped["sqft"].isna(), ["listing_id", "bedrooms"]].copy()
options["overall_median"] = overall_median
options["same_bedroom_median"] = bedroom_median[options.index]
display(options)

print("Listings kept if we drop blank sizes:", deduped["sqft"].notna().sum(), "of", len(deduped))
print("Observed studio sizes:", deduped.loc[deduped["bedrooms"] == 0, "sqft"].dropna().tolist())

,listing_id,bedrooms,overall_median,same_bedroom_median
3,L04,2,600.0,775.0
9,L10,1,600.0,550.0
11,L12,3,600.0,1450.0
12,L13,0,600.0,375.0


Listings kept if we drop blank sizes: 9 of 13
Observed studio sizes: [400.0, 350.0]


| Choice | What it does here | Risk |
|---|---|---|
| Drop rows | Keeps only listings with a size | Loses 4 of 13 listings, including a \$3,200 Fenway 2-bedroom |
| Overall median | Every blank gets 600 sq ft | Studio L13 gets more space than any real studio in the data |
| Median for the same bedroom count | A blank studio gets the typical studio size | Groups have only 2–3 listings, so one odd unit moves the value |

We use the same-bedroom median and **keep a flag** so every filled size can be traced. In Part 6 the flag also lets us redo this fill for the model, using training rows only.

### E4 · Fill size by bedroom count, and keep a flag
Start from `deduped`.
1. `filled["sqft_was_missing"]`: `True` where size is blank. **Make the flag before filling.**
2. Fill blank `sqft` with the median size of listings with the same bedroom count.
3. Leave `rent` alone.

In [14]:
filled = None

need(deduped, "the Part 3 duplicates example")

# Write your code below. Your code must create the variable set to None above.

# 1. filled: a copy of deduped.
filled = deduped.copy()

# 2. filled["sqft_was_missing"]: True where sqft is blank. Make this flag BEFORE filling.
filled["sqft_was_missing"] = filled["sqft"].isna()

# 3. same_bedroom_median: for every row, the median sqft of the listings with the
#    same bedroom count.
#    Hint: .groupby("bedrooms")["sqft"].transform("median")
same_bedroom_median = filled.groupby("bedrooms")["sqft"].transform("median")

# 4. Fill the blank sqft values with same_bedroom_median.
#    Hint: .fillna()
filled["sqft"].fillna(same_bedroom_median, inplace=True)

# 5. Display listing_id, bedrooms, sqft, sqft_was_missing, and rent.
display(filled[["listing_id", "bedrooms", "sqft", "sqft_was_missing", "rent"]])

/tmp/ipykernel_13433/1262862353.py:20: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  filled["sqft"].fillna(same_bedroom_median, inplace=True)


,listing_id,bedrooms,sqft,sqft_was_missing,rent
0,L01,2,750.0,False,2900.0
1,L02,0,400.0,False,1850.0
2,L03,1,520.0,False,2450.0
3,L04,2,775.0,True,3200.0
4,L05,3,1800.0,False,9500.0
5,L06,2,800.0,False,2600.0
6,L07,3,1100.0,False,3150.0
7,L08,1,600.0,False,2100.0
8,L09,0,350.0,False,NaN
9,L10,1,550.0,True,2050.0


**Check E4**

In [15]:
need(filled, "E4")
check("no size is blank", filled["sqft"].notna().all())
check("the flag marks exactly the sizes that were blank", filled["sqft_was_missing"].equals(deduped["sqft"].isna()))
expected = deduped.groupby("bedrooms")["sqft"].transform("median")
check("filled sizes equal the same-bedroom median",
      np.allclose(filled.loc[filled["sqft_was_missing"], "sqft"], expected[filled["sqft_was_missing"]]))
check("rent was not filled", filled["rent"].isna().sum() == deduped["rent"].isna().sum())

PASS  no size is blank
PASS  the flag marks exactly the sizes that were blank
PASS  filled sizes equal the same-bedroom median
PASS  rent was not filled


**Write:** L10's size was 0 and is now 550. What does 550 mean, and what does it not mean? What does the flag tell a reader?

*Your answer:*
550 is the median square footage of every other 1-bedroom apartments.
It does not mean that it is actual physical size of L10
It tells the reader that sqft_was_missing was True

### AI check 2 · Filling the sizes
Ask your AI tool:
> I have 13 apartment listings. Four have no size in square feet, and one of those four is a studio. Should I fill the blanks with the overall median size?

Paste its main recommendation, then judge it with the class.

| Question | Your note |
|---|---|
| Did it notice that a studio is smaller than most apartments? | Yes |
| Did it suggest marking the filled values (a flag)? | Yes |
| Did it ask how the data will be used? | Yes |
| Did it give a number without seeing our data? | No |

*AI answer:*

No, you should not fill the blanks with the overall median size.
Using an overall median will skew the missing studio's value because larger units (1-, 2-, and 3-bedroom listings) pull the overall figure up, resulting in a size much larger than any typical studio.
Instead, use group-level median imputation:
•	Calculate and apply the median square footage grouped by the number of bedrooms (so the studio receives the median of other studios).
•	Add a boolean indicator column (e.g., sqft_was_missing) before imputing so downstream analyses and models can distinguish between true measured sizes and filled estimates.
How do you plan to use this data—is it primarily for summary reporting, or will it serve as training data for a predictive model? If sample sizes per bedroom count are tiny (e.g., only one or two studios exist), dropping missing rows might be preferable depending on your end goal.

# Part 5 · Reduce
Drop a column only when you can say why. Two quick tests:

In [16]:
need(filled, "E4")
print("Number of different values in each column:")
display(filled.nunique())
print("Correlation of sqft and size_m2:")
display(listings_raw[["sqft", "size_m2"]].corr().round(3))

Number of different values in each column:


,0
listing_id,13
posted,11
neighborhood,6
bedrooms,4
rent_text,13
sqft,12
city,1
size_m2,10
walk_minutes,5
_merge,2


Correlation of sqft and size_m2:


,sqft,size_m2
sqft,1.0,1.0
size_m2,1.0,1.0


- `city` has one value in every row, so it tells us nothing.
- `size_m2` is `sqft × 0.0929`. The correlation of 1.00 confirms what the formula already tells us. Keeping it would also mean cleaning it (L10 is 0 m² there too). A high correlation between two **different** measurements would **not** be enough reason to delete one.
- `rent_text` has been replaced by `rent`.
- `listing_id` and `posted` stay, for tracing, even though they will not be model inputs.

### E5 · Build the report table
`report_table`: `filled` without `city`, `size_m2`, and `rent_text`.

In [17]:
report_table = filled.copy()

# Write your code below. Your code must create the variable set to None above.

# 1. report_table: filled without the columns city, size_m2, and rent_text.
#    Hint: .drop(columns=[...])
report_table = report_table.drop(columns=["city", "size_m2", "rent_text"])

# 2. Display report_table.
display(report_table)

,listing_id,posted,neighborhood,bedrooms,sqft,walk_minutes,_merge,rent,rent_was_invalid,sqft_was_invalid,sqft_was_missing
0,L01,2026-08-03,Mission Hill,2,750.0,10.0,both,2900.0,False,False,False
1,L02,2026-08-05,Mission Hill,0,400.0,10.0,both,1850.0,False,False,False
2,L03,2026-08-06,Fenway,1,520.0,12.0,both,2450.0,False,False,False
3,L04,2026-08-06,Fenway,2,775.0,12.0,both,3200.0,False,False,True
4,L05,2026-08-07,Back Bay,3,1800.0,25.0,both,9500.0,False,False,False
5,L06,2026-08-08,Allston,2,800.0,45.0,both,2600.0,False,False,False
6,L07,2026-08-20,Jamaica Plain,3,1100.0,30.0,both,3150.0,False,False,False
7,L08,2026-08-09,Brighton,1,600.0,NaN,left_only,2100.0,False,False,False
8,L09,2026-08-10,Jamaica Plain,0,350.0,30.0,both,NaN,True,False,False
9,L10,2026-08-11,Allston,1,550.0,45.0,both,2050.0,False,True,True


**Check E5**

In [18]:
need(report_table, "E5")
check("city, size_m2, and rent_text are gone", not {"city", "size_m2", "rent_text"} & set(report_table.columns))
check("no rows were removed", len(report_table) == len(filled))
check("listing_id and the flags are still there",
      {"listing_id", "sqft_was_missing", "rent_was_invalid", "sqft_was_invalid"} <= set(report_table.columns))

PASS  city, size_m2, and rent_text are gone
PASS  no rows were removed
PASS  listing_id and the flags are still there


**Write:** name one column you kept even though it will not be a model input, and say why.

*Your answer:*
listing_id, it is a unique identifier to ensure no data gets mixed up.

### Worked example · Report snapshot

In [19]:
need(report_table, "E5")
summary = (report_table.groupby("neighborhood")
           .agg(listings=("listing_id", "count"),
                median_rent=("rent", "median"),
                walk_minutes=("walk_minutes", "first"))
           .sort_values("walk_minutes"))
display(summary)
print("Listings within a 15-minute walk:", (report_table["walk_minutes"] <= 15).sum())
print("Listings with unknown walking time:", report_table["walk_minutes"].isna().sum())

,listings,median_rent,walk_minutes
neighborhood,,,
Mission Hill,3,2900.0,10.0
Fenway,3,2450.0,12.0
Back Bay,2,6400.0,25.0
Jamaica Plain,2,3150.0,30.0
Allston,2,2325.0,45.0
Brighton,1,2100.0,NaN


Listings within a 15-minute walk: 6
Listings with unknown walking time: 1


**Talk about it:** Back Bay's median rent comes from only two listings, and one of them is the \$9,500 unit. Brighton's walking time is unknown, which is not the same as "far". A good report says both.

# Part 6 · Transform
### Encoding: does the category have an order?
- **Neighborhood has no order.** Fenway is not "more" than Allston. Codes like Allston = 1, Fenway = 2 would invent an order and a distance, so we use **one-hot encoding**: one 0/1 column per neighborhood.
- **Bedrooms has a real order:** studio < 1 < 2 < 3. Keeping it as 0, 1, 2, 3 is reasonable.

In [20]:
display(pd.get_dummies(pd.DataFrame({"neighborhood": ["Fenway", "Allston", "Back Bay"]}), dtype=int))

,neighborhood_Allston,neighborhood_Back Bay,neighborhood_Fenway
0,0,0,1
1,1,0,0
2,0,1,0


### Learn from the training rows only
Some steps apply a **fixed rule**, such as "remove `$` and commas" or "a rent of \$0 is invalid". A fixed rule gives the same result whatever rows you use, so it can run before the split.

Other steps **learn numbers from the data**. For a model, they must learn from the **training rows only**. Otherwise the test rows help build the model, and the test is no longer a fair check.

| Step | What it learns |
|---|---|
| Fill blank sizes | The median size for each bedroom count |
| `StandardScaler` | The mean and standard deviation of each column |
| `OneHotEncoder` | The list of neighborhoods |

For every step, ask: **what did it learn, and from which rows?**

Part 4 filled sizes using all 13 listings. That is fine for the report. For the model, we blank those filled sizes again (the flag says which ones) and refill them from the training rows.

### E6 · Build the model inputs
The cell gives you `X`, `y`, and three helper functions. `X` keeps `listing_id` for tracing; `prepare()` leaves it out of the inputs.

1. Split with `test_size=0.25` and `random_state=RANDOM_STATE`
2. `train_medians`: the median `sqft` for each bedroom count, from the **training** rows
3. `scaler`: a `StandardScaler` fitted on the filled **training** `bedrooms` and `sqft`
4. `encoder`: fitted on the **training** neighborhoods
5. `X_train_ready` and `X_test_ready`: made with `prepare()`

In [21]:
X_train = X_test = y_train = y_test = None
train_medians = scaler = encoder = None
X_train_ready = X_test_ready = None

need(report_table, "E5")
numeric_cols = ["bedrooms", "sqft"]

model_rows = report_table[report_table["rent"].notna()].copy()                # L09 has no rent
model_rows["sqft"] = model_rows["sqft"].mask(model_rows["sqft_was_missing"])  # undo Part 4's fill
X = model_rows[["listing_id", "bedrooms", "sqft", "neighborhood"]]
y = model_rows["rent"]

def make_encoder():
    """One 0/1 column per neighborhood. A neighborhood not seen in training gets all zeros."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:  # older scikit-learn
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def fill_sqft(table):
    """Fill blank sizes with the training median for the same bedroom count."""
    out = table.copy()
    out["sqft"] = out["sqft"].fillna(out["bedrooms"].map(train_medians))
    return out

def prepare(table):
    """Fill, scale, and encode rows using only what was learned from the training rows."""
    numbers = pd.DataFrame(scaler.transform(fill_sqft(table)[numeric_cols]),
                           columns=numeric_cols, index=table.index)
    hoods = pd.DataFrame(encoder.transform(table[["neighborhood"]]),
                         columns=encoder.get_feature_names_out(), index=table.index)
    return pd.concat([numbers, hoods], axis=1)

# Write your code below. Your code must create the variables set to None above.
# X, y, fill_sqft(), make_encoder(), and prepare() are given above.

# 1. Split X and y into X_train, X_test, y_train, y_test.
#    Use test_size=0.25 and random_state=RANDOM_STATE.
#    Hint: train_test_split()
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE)


# 2. train_medians: the median sqft for each bedroom count, from X_train ONLY.
#    Hint: .groupby("bedrooms")["sqft"].median()
train_medians = X_train.groupby("bedrooms")["sqft"].median()

# 3. scaler: a StandardScaler fitted on the filled training sizes and bedrooms.
#    Hint: StandardScaler().fit(fill_sqft(X_train)[numeric_cols])
scaler = StandardScaler().fit(fill_sqft(X_train)[numeric_cols])

# 4. encoder: make_encoder() fitted on the training neighborhoods, X_train[["neighborhood"]].
encoder = make_encoder().fit(X_train[["neighborhood"]])

# 5. X_train_ready and X_test_ready: use prepare() on X_train and on X_test.
X_train_ready = prepare(X_train)
X_test_ready = prepare(X_test)

# 6. Print how many rows are in X, X_train, and X_test. Display X_test_ready.round(2).
print("Rows in X:", len(X))
print("Rows in X_train:", len(X_train))
print("Rows in X_test:", len(X_test))
display(X_test_ready.round(2))

Rows in X: 12
Rows in X_train: 9
Rows in X_test: 3


,bedrooms,sqft,neighborhood_Allston,neighborhood_Back Bay,neighborhood_Brighton,neighborhood_Fenway,neighborhood_Jamaica Plain,neighborhood_Mission Hill
11,1.46,1.61,0.0,0.0,0.0,0.0,0.0,1.0
10,-0.42,-0.54,0.0,1.0,0.0,0.0,0.0,0.0
0,0.52,-0.06,0.0,0.0,0.0,0.0,0.0,1.0


**Check E6**

In [22]:
need(X_test_ready, "E6")
check("the unknown rent is not in the model rows",
      y.notna().all() and len(X) == report_table["rent"].notna().sum())
check("train and test share no listings", not set(X_train["listing_id"]) & set(X_test["listing_id"]))
check("size medians came from the training rows",
      train_medians.equals(X_train.groupby("bedrooms")["sqft"].median()))
check("the scaler learned from the filled training rows",
      np.allclose(scaler.mean_, fill_sqft(X_train)[numeric_cols].mean()))
check("the encoder learned the training neighborhoods",
      list(encoder.categories_[0]) == sorted(X_train["neighborhood"].unique()))
check("X_test_ready holds the test rows", X_test_ready.index.equals(X_test.index))
check("train and test inputs have the same columns", list(X_train_ready.columns) == list(X_test_ready.columns))
check("rent and listing_id are not inputs", not {"rent", "listing_id"} & set(X_test_ready.columns))
check("no blanks remain in the inputs",
      not X_train_ready.isna().any().any() and not X_test_ready.isna().any().any())

PASS  the unknown rent is not in the model rows
PASS  train and test share no listings
PASS  size medians came from the training rows
PASS  the scaler learned from the filled training rows
PASS  the encoder learned the training neighborhoods
PASS  X_test_ready holds the test rows
PASS  train and test inputs have the same columns
PASS  rent and listing_id are not inputs
PASS  no blanks remain in the inputs


**Write:** why does `bedrooms` stay one column while `neighborhood` becomes six?

*Your answer:*
Because it is numeric and ordered.

### Worked example · What changes if the test rows help fit the scaler?

In [23]:
need(X_test_ready, "E6")
leaky_scaler = StandardScaler().fit(fill_sqft(X)[numeric_cols])   # wrong: the test rows help set the mean
comparison = pd.DataFrame({
    "listing_id": X_test["listing_id"],
    "sqft": fill_sqft(X_test)["sqft"],
    "scaled_train_only": X_test_ready["sqft"].round(2),
    "scaled_with_test_rows": leaky_scaler.transform(fill_sqft(X_test)[numeric_cols])[:, 1].round(2),
})
display(comparison)

,listing_id,sqft,scaled_train_only,scaled_with_test_rows
11,L12,1450.0,1.61,1.54
10,L11,550.0,-0.54,-0.63
0,L01,750.0,-0.06,-0.15


The two columns differ because the test rows moved the mean and spread. With 12 rows the gap is small. The rule still matters: the test rows must stay unseen until the final check.

### Worked example · A neighborhood the model has never seen
Two new listings need a suggested rent. N01 is in **Dorchester**, which never appeared in training. (Real new listings would first go through the same Part 3 cleaning rules.)

**Predict:** what will N01's six neighborhood columns look like?

*Your prediction:*
the neighborhood columns will be 0 for N01

In [24]:
need(X_test_ready, "E6")
new_listings = pd.DataFrame({
    "listing_id": ["N01", "N02"],
    "bedrooms": [2, 0],
    "sqft": [850, np.nan],
    "neighborhood": ["Dorchester", "Fenway"],
})
print("get_dummies on the new rows alone:")
display(pd.get_dummies(new_listings[["neighborhood"]], dtype=int))
print("The fitted encoder, with the training columns:")
display(prepare(new_listings).round(2))

get_dummies on the new rows alone:


,neighborhood_Dorchester,neighborhood_Fenway
0,1,0
1,0,1


The fitted encoder, with the training columns:


,bedrooms,sqft,neighborhood_Allston,neighborhood_Back Bay,neighborhood_Brighton,neighborhood_Fenway,neighborhood_Jamaica Plain,neighborhood_Mission Hill
0,0.52,0.18,0.0,0.0,0.0,0.0,0.0,0.0
1,-1.36,-0.90,0.0,0.0,0.0,1.0,0.0,0.0


`get_dummies` only knows the rows you give it, so its columns don't match the six the model was trained on. The fitted encoder keeps the training columns.

Dorchester gets **all zeros**, meaning "none of the neighborhoods I know". That is better than crashing, but the model learned nothing about Dorchester, so a suggested rent for N01 needs a warning. N02's blank size was filled from the training studios.

# Exit ticket
Answer without running more code.
1. A join ran with no error, but five listings got no walking time. What would you check first?
2. Listing L13 is a studio with no size. Why is 375 sq ft a better fill than 600, and what does its flag tell a reader?
3. In E6, three things were learned from the data. Name them, and say which rows they had to come from.

*Your answers:*
1. First, I would check to. ensure the join keys were mismatched
2. 375 is the median of sqft, the average is higher because of large outliers
3. Median square footage for bedroom count (train_medians), came from unfilled training rows.

Column means and standard deviations (scaler), laerned from the filled training rows (fill_sqft(X_train)).

Set of unique neighborhood categories (encoder), came from training rows (X_train[["neighborhood"]]).

# Submit
- Restart the kernel and run all cells. Every check should print `PASS`.
- Every *Your answer* and *Your prediction* cell is filled in.
- Both AI checks include the AI's answer and your notes.
- Submit this notebook with its outputs.